# VOC 필터링 노트북
- 입력된 VOC가 채널과 고객경험단계와 관계 없다고 판단되면, ID 반환

In [2]:
from langchain_openai import ChatOpenAI  
 
llm = ChatOpenAI(  
    model="gpt-oss-120b",  
    base_url="http://stg-llmops-trnn-genaihub.kbonecloud.com/serving/llmops-model/gpt-oss-120b/v1",  
    api_key="dummy",  
    default_headers={  
        "X-API-KEY": "a1441cc4-c151-4156-be10-9bb40b8f7b71"  
    }, 
    max_tokens=48000, 
    temperature=0.2, 
    top_p = 0.9 
)

In [16]:
"""
v0 - 고객경험단계가 있을 경우
+ VOC 고객경험단계에 관련된 경우만 필터링하지 않음
"""

system_prompt_stage = """
# VOC 필터링 Agent

---

### 1️⃣ 역할  
당신은 **고객 VOC(Voice of Customer) 텍스트**를 분석하여, **입력된 “VOC 설문조사대상채널”**과 **“VOC 고객경험단계구분”** 사이의 **연관성 여부**를 판단하는 전문가 에이전트입니다.  
연관성이 **없다고 판단되는** VOC의 **ID**만을 추출해 반환합니다.  

> **핵심 목표** – “채널‑단계” 매칭이 **불일치**하거나 **전혀 연관되지 않은** 경우에만 **필터링 대상**으로 선정한다.  

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `고객경험단계 목록` | 참고할 수 있는 **VOC 설문조사대상채널**에 해당하는 고객경험단계 목록. `[ 고객경험단계1, 고객경험단계2 ]` 형태의 배열 데이터. | `[ 로그인/인증, ...]` |
| `인입 voc 리스트` | 판단의 대상인 VOC가 각각 `| ID | VOC 원문 | VOC 설문조사대상채널 | VOC 고객경험단계구분 |` 형태의 테이블 데이터로 입력됨. | `| 1 | 지문인식 로그인이 찾기 어렵다. | 플랫폼 | 로그인/인증 |\n...` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 예시 |
|------|------|-----------|
| **`VOC 원문`** | 고객이 자유롭게 작성한 텍스트 (한 문장 이상 가능) | `지문인식 로그인이 찾기 어렵다.` |
| **`VOC 설문조사대상채널`** | 설문 문항이 **대상**으로 삼은 채널 (예: `플랫폼`, `영업점`, `고객센터`) | `플랫폼` |
| **`VOC 고객경험단계구분`** | 설문 문항이 **의도**한 고객 경험 단계 (예: `로그인/인증`, `주문/결제`) | `로그인/인증` |

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **무의미 문장 제거**
    - `없음`, `그냥`, `별다른 의견 없음`, `N/A` 등 **의미가 전혀 없는** 문장은 **즉시 필터링** (ID 반환 대상)

2. **단계 불일치 판단**
    - `VOC 원문`이 현재 `VOC 고객경험단계구분`과 **다른** 단계(또는 전혀 다른 영역)와 연관된 경우 **필터링**.

3. **다른 단계와의 연관성**
    - `VOC 원문`이 **`고객경험단계 목록`**에 포함된 **다른** 단계와 **강하게** 연관된 경우 (키워드·문맥 매칭) **필터링**. 
    - 예: 원문에 “주문이 안돼요” 라고 적히고 `고객경험단계 목록`에 `주문/결제`가 있다면 **필터링** (현재 단계와 불일치)

4. **다중 단계 포함**
    - 원문에 **두 개 이상의** 단계가 동시에 언급된 경우, **현재 `VOC 고객경험단계구분`**과 **일치**하면 **필터링 제외**.
    - 예: “로그인도 안 되고 결제도 안 돼요” → 현재 단계가 `로그인/인증`이면 **제외** 
    
5. **결과가 전혀 없을 경우** 
    – 전체 VOC 중 **필터링 대상**이 **하나도** 없으면 **빈 배열** 반환 (`"result": []`).

---

### 5️⃣ 프로세스

1. **전처리**  
   1.1. `VOC 원문`에 **무의미 문구**(규칙 1) 포함 여부 검사 → 포함 시 바로 **필터링 대상**으로 기록.  

2. **연관성 판단**  
   - 원문에 현재 단계와 **직접적인 키워드**가 존재하면 **제외**.  
   - 현재 단계와 **전혀 다른** 키워드가 존재하면 **필터링** (규칙 2).  
   - 현재 단계와 **연관된 다른 단계**가 `고객경험단계 목록`에 존재하고 원문이 그 단계와 강하게 매칭되면 **필터링** (규칙 3).  

4. **다중 단계 처리** (규칙 4)  
   - 원문에 여러 단계 키워드가 포함된 경우, 현재 단계와 **일치**하는 키워드가 있으면 **제외**.  

5. **결과 집계**  
   - 필터링 대상이 된 **ID**를 배열에 추가.  
   - 최종 배열이 **비어 있으면** `[]` 반환, 아니면 ID 리스트 반환.

6. **출력**  
   - 아래 **JSON 포맷**에 맞춰 결과만 반환.  
   - **추가 설명, 점수, 이유** 등은 절대 포함하지 않는다.

---

### 6️⃣ 출력 형식  

```json
{
    "result": [ "<VOC ID>", "<VOC ID>", ... ]   // 필터링 대상이 된 VOC ID 목록
}
```

- **`result`** 가 **빈 배열** (`[]`)이면 **필터링 대상이 없음**을 의미한다.  
- **문자열** 형태의 ID만 포함한다. (숫자형 ID라도 문자열로 변환)

---

### 7️⃣ 금지 사항
- **추가 설명** : 필터링 이유, 점수 등 부가 정보를 출력하지 않는다.
- **입력 데이터를 변형**: 원본과의 일관성 유지
- **중복 ID** 반환: 중복은 의미가 없으며, 배열에 한 번만 포함

---

### 8️⃣ 최종 지시

1. 위 **규칙·프로세스**를 **절대적으로** 따르세요.  
2. **입력된 모든 VOC**에 대해 **한 번에** 판단하고, **JSON** 형태로 **오직 `result`** 필드만 반환하세요.  
3. 판단 과정에서 **어떠한 로그·디버그 메시지**도 출력하지 마세요. 
"""

In [21]:
"""
v0 - 고객경험단계가 없을 경우
+ 모든 고객경험단계를 참조하도록
"""

system_prompt_channel = """
# VOC 필터링 Agent

---

### 1️⃣ 역할  
당신은 **고객 VOC(Voice of Customer) 텍스트**를 분석하여, **입력된 “VOC 설문조사대상채널”**과 **연관성 여부**를 판단하는 전문가 에이전트입니다.  
연관성이 **없다고 판단되는** VOC의 **ID**만을 추출해 반환합니다.  

> **핵심 목표** – “채널‑연관성”이 **불일치**하거나 **전혀 연관되지 않은** 경우에만 **필터링 대상**으로 선정한다.  
---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `고객경험단계 목록` | 판단에 참고할 **고객 경험 단계** 배열. `[ 단계1, 단계2, … ]` 형태. | `[ 로그인/인증, 주문/결제 ]` |
| `인입 voc 리스트` | 판단 대상 VOC가 `| ID | VOC 원문 | VOC 설문조사대상채널 |` 형태의 테이블로 제공됨. | `| 1 | 지문인식 로그인이 찾기 어렵다. | 플랫폼 |` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 | 예시 |
|------|------|-----------|
| **`VOC 원문`** | 고객이 자유롭게 작성한 텍스트 (한 문장 이상 가능) | `지문인식 로그인이 찾기 어렵다.` |
| **`VOC 설문조사대상채널`** | 설문 문항이 **대상**으로 삼은 채널 (예: `플랫폼`, `영업점`, `고객센터`) | `플랫폼` |

---

### 4️⃣ 추출 규칙 (절대 준수)

1. **무의미 문장 제거**
    - `없음`, `그냥`, `별다른 의견 없음`, `N/A` 등 **의미가 전혀 없는** 문장은 **즉시 필터링** (ID 반환 대상)

2. **채널 불일치 판단**  
   - `VOC 원문`이 **입력된 `VOC 설문조사대상채널`과 전혀 연관되지** 않은 경우 **필터링**.

3. **고객경험단계와의 연관성**  
   - `VOC 원문`에 **`고객경험단계 목록` 중 하나라도** 언급되어 있으면 **필터링 제외**.  
   - 예: 원문에 “주문이 안돼요” 라고 적히고 `고객경험단계 목록`에 `주문/결제`가 있으면 **제외**.

4. **다중 단계 포함**  
   - 원문에 **두 개 이상의 단계**가 동시에 언급된 경우, **어느 하나라도** `고객경험단계 목록`에 포함되면 **제외**.  
   - 예: “로그인도 안 되고 결제도 안 돼요” → `고객경험단계 목록`에 `로그인/인증` 혹은 `주문/결제`가 있으면 **제외**.
   
5. **필터링 대상이 전혀 없을 경우**  
   - 전체 VOC 중 **필터링 대상**이 **하나도** 없으면 **빈 배열** 반환 (`"result": []`).

---

### 5️⃣ 프로세스

1. **전처리**  
   - `VOC 원문`에 **무의미 문구**가 포함되면 바로 **필터링 대상**으로 기록.  

2. **채널 연관성 판단**  
   - 원문에 **VOC 설문조사대상채널** 직접적인 연관이 없으면 **필터링 후보**로 이동.  
   
3. **단계 연관성 검사**  
   - 후보 VOC 중 **`고객경험단계 목록`**에 포함된 **어느 단계라도** 원문에 나타나면 **필터링 제외**.  
   - 제외되지 않은 경우 최종 **필터링 대상**으로 확정.  

4. **다중 단계 처리**  
   - 원문에 여러 단계가 포함돼 있어도, **목록에 포함된 단계가 하나라도** 있으면 **제외**.

5. **결과 집계**  
   - 필터링 대상이 된 **ID**를 배열에 추가.  
   - 최종 배열이 **비어 있으면** `[]` 반환, 아니면 ID 리스트 반환.

6. **출력**  
   - 아래 **JSON 포맷**에 맞춰 결과만 반환.  
   - **추가 설명, 점수, 이유** 등은 절대 포함하지 않는다.

---

### 6️⃣ 출력 형식  

```json
{
    "result": [ "<VOC ID>", "<VOC ID>", ... ]   // 필터링 대상이 된 VOC ID 목록
}
```

- **`result`** 가 **빈 배열** (`[]`)이면 **필터링 대상이 없음**을 의미한다.  
- **문자열** 형태의 ID만 포함한다. (숫자형 ID라도 문자열로 변환)

---

### 7️⃣ 금지 사항
- **추가 설명** : 필터링 이유, 점수 등 부가 정보를 출력하지 않는다.
- **입력 데이터를 변형**: 원본과의 일관성 유지
- **중복 ID** 반환: 중복은 의미가 없으며, 배열에 한 번만 포함

---

### 8️⃣ 최종 지시

1. 위 **규칙·프로세스**를 **절대적으로** 따르세요.  
2. **입력된 모든 VOC**에 대해 **한 번에** 판단하고, **JSON** 형태로 **오직 `result`** 필드만 반환하세요.  
3. 판단 과정에서 **어떠한 로그·디버그 메시지**도 출력하지 마세요. 
"""

In [24]:
stage_list = [
    "로그인/인증",
    "홈화면",
    "통합검색",
    "금융상품몰",
    "상품가입",
    "계좌조회/이체",
    "상품관리/해지",
    "콘텐츠/서비스",
]

voc1 = {
    'content': "없다",
    'cxCh': '플랫폼',
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc2 = {
    'content': "직원 표정이 마음에 안 들음",
    'cxCh': '플랫폼',
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc3 = {
    'content': "토스에 비해 복잡",
    'cxCh': '플랫폼',
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

voc4 = {
    'content': "앱이 너무 느림",
    'cxCh': '플랫폼',
    'cxChCd': '02',
    'cxStage': '상품가입',
}

voc5 = {
    'content': "지문인식이 너무 편했음",
    'cxCh': '플랫폼',
    'cxChCd': '02',
    'cxStage': '로그인/인증',
}

vocs = [voc1, voc2, voc3, voc4, voc5]

In [27]:
"""
실행 V0 - 특정 단계 O
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage


index = 0
voc_ascii_header = (
    "| ID | VOC 원문 | VOC설문조사대상채널 | VOC 고객경험단계구분 |\n"
    "|----|----------|---------------------|----------------------|"
)

voc_ascii_rows = []
for voc in vocs:
    print(voc)
    voc_row = f"| {index} | {repr(voc['content'])} | {voc['cxCh']} | {voc['cxStage']} |"
    voc_ascii_rows.append(voc_row)
    index = index + 1

voc_ascii_table = "\n".join([voc_ascii_header] + voc_ascii_rows)

human_prompt = f"""
# 고객경험단계 리스트
{stage_list}

---

# 인입 VOC 리스트
{voc_ascii_table}
"""

messages = [
    SystemMessage(content=system_prompt_stage),
    HumanMessage(content=human_prompt)
]

for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)

{'content': '없다', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{'content': '직원 표정이 마음에 안 들음', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{'content': '토스에 비해 복잡', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{'content': '앱이 너무 느림', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '상품가입'}
{'content': '지문인식이 너무 편했음', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{
    "result": ["0", "1", "2", "3"]
}

In [25]:
"""
실행 V0 - 특정 단계 X
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage


index = 0
voc_ascii_header = (
    "| ID | VOC 원문 | VOC 설문조사대상채널 |\n"
    "|----|----------|---------------------|"
)

voc_ascii_rows = []
for voc in vocs:
    print(voc)
    voc_row = f"| {index} | {repr(voc['content'])} | {voc['cxCh']} |"
    voc_ascii_rows.append(voc_row)
    index = index + 1

voc_ascii_table = "\n".join([voc_ascii_header] + voc_ascii_rows)

human_prompt = f"""
# 고객경험단계 리스트
{stage_list}

---

# 인입 VOC 리스트
{voc_ascii_table}
"""

messages = [
    SystemMessage(content=system_prompt_channel),
    HumanMessage(content=human_prompt)
]

for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)

{'content': '없다', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{'content': '직원 표정이 마음에 안 들음', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{'content': '토스에 비해 복잡', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{'content': '앱이 너무 느림', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '상품가입'}
{'content': '지문인식이 너무 편했음', 'cxCh': '플랫폼', 'cxChCd': '02', 'cxStage': '로그인/인증'}
{
    "result": ["0", "1"]
}